# Predicción de Consumo Eléctrico Horario

Proyecto de regresión / series de tiempo listo para **Google Colab**.

**Pregunta:** ¿podemos predecir el consumo eléctrico de la próxima hora?

**Target:** consumo

**Stack:** pandas, seaborn, scikit-learn, shap

Datos simulados de consumo horario.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Generar serie de consumo horario

In [ ]:
np.random.seed(RANDOM_STATE)
n = 1000
hour = np.arange(n)
# Ciclo diario + semanal
daily = 20 * np.sin(2 * np.pi * hour / 24)
weekly = 10 * np.sin(2 * np.pi * hour / (24*7))
noise = np.random.normal(0, 5, n)
consumo = 100 + daily + weekly + noise
df = pd.DataFrame({
    "hour": hour,
    "consumo": consumo,
    "temp": 20 + 5 * np.sin(2 * np.pi * hour / 24) + np.random.normal(0, 2, n),
    "is_weekend": ((hour // 24) % 7 >= 5).astype(int),
})
# Features lag
df["consumo_lag1"] = df["consumo"].shift(1)
df["consumo_lag24"] = df["consumo"].shift(24)
df["ma24"] = df["consumo"].rolling(24).mean()
df = df.dropna().reset_index(drop=True)
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.describe().T)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["hour"].iloc[:200], df["consumo"].iloc[:200])
plt.title("Consumo eléctrico (primeras 200 horas)")
plt.xlabel("Hora")
plt.ylabel("Consumo")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="temp", y="consumo", ax=axes[0])
axes[0].set_title("Temperatura vs Consumo")
sns.boxplot(data=df, x="is_weekend", y="consumo", ax=axes[1])
axes[1].set_title("Consumo fin de semana")
plt.tight_layout()
plt.show()

### Hallazgos
- Ciclo diario claro.
- Temperatura y lags correlacionan.
- Ideal para forecast de demanda energética.

## 3. Preparar datos

In [ ]:
X = df.drop(columns=["consumo", "hour"])
y = df["consumo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred, squared=False),
        "R2": r2_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("R2", ascending=False)

## 5. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Importancia — Random Forest")
plt.tight_layout()
plt.show()

## 6. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X.columns, max_display=6, show=False)
plt.title("SHAP Summary — Consumo")
plt.tight_layout()
plt.show()

## 7. Conclusiones

- Los lags de 1 y 24 horas son los más importantes.
- Temperatura aporta señal.
- Se puede extender a datos reales de smart meters.

**Rango esperado (ejemplo):** R² > 0.75